# Урок 20 — Контекстні менеджери: архітектура гарантованого життєвого циклу

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Як розробники, ми маємо виходити з базового припущення:
**будь-яка взаємодія із зовнішнім ресурсом рано чи пізно завершиться помилкою.**

Мережеві з’єднання обриваються.
Бази даних відхиляють запити.
Користувачі та розробники передають невалідні дані.

Якщо ви покладаєтесь на ручне звільнення ресурсів — виклики `.close()`, `.release()` або `.rollback()` —
ви неминуче зіткнетеся з **помилками через пропущене очищення ресурсів (errors of omission)**:
багами, коли розробник просто *забуває* виконати необхідне завершення роботи з ресурсом.

---

### Ключова ідея

**Контекстний менеджер — це відповідь Python на цю проблему.**

Він перетворює роботу з ресурсом на чітко визначену **область виконання (scope)**:

* при вході в область — ресурс **захоплюється (acquire)**
* при виході — ресурс **гарантовано звільняється (release)**

Незалежно від того, чи виникла помилка.

---

## Структура уроку

| Частина | Патерн                             | Ключова ідея                           |
| ------- | ---------------------------------- | -------------------------------------- |
| 1       | `try...finally` → `with`           | Еволюція управління ресурсами          |
| 2       | Протокол: `__enter__` / `__exit__` | Механіка роботи контекстних менеджерів |
| 3       | Клас-менеджер: `ListTransaction`   | Патерн транзакції                      |
| 4       | Клас-менеджер: `RedirectStdout`    | Патерн тимчасової зміни стану          |
| 5       | `@contextmanager`                  | Генератори як контекстні менеджери     |
| 6       | Lock Pattern                       | Уникнення race condition і блокувань   |
| 7       | Transaction Pattern                | «Все або нічого»                       |
| 8       | `__exit__` як контролер            | Управління та придушення винятків      |
| 9       | `contextlib.suppress`              | Практика зі стандартної бібліотеки     |

---


## Частина 1: Від `try...finally` до `with` — Еволюція управління ресурсами

---

### Передбачення

> **Питання:** Ви відкрили файл і отримали виняток посередині запису.  
> Файл закрився?
> Скільки разів таке може трапитись, перш ніж сервер впаде з `OSError: Too many open files`?

### Пояснення: Крихкість ручного управління ресурсами

У традиційному підході (без контекстних менеджерів) управління ресурсами є **багатослівним і крихким**.
Розробники змушені явно контролювати життєвий цикл ресурсу через блок `try...finally`:

```python
f = open('hello.txt', 'w')
try:
    f.write('привіт, світ!')
finally:
    f.close()  # Виконається ЗАВЖДИ — навіть якщо write() викличе виняток
```

Цей підхід **покладається на дисципліну розробника** і легко ламається:

* можна **забути `finally`**
* можна **закрити ресурс не у всіх гілках логіки**
* код стає **шумним і важче читається**

У результаті виникають типові проблеми:

* витоки ресурсів (file descriptors, sockets)
* завислі транзакції в БД
* блокування (locks), які не були звільнені

---

### Рішення: `with`

Інструкція `with` замінює цей шаблон, **інкапсулюючи управління ресурсом у самому об’єкті**:

```python
with open('hello.txt', 'w') as f:
    f.write('привіт, світ!')
# Файл гарантовано закрито тут, навіть якщо виник виняток
```

Що відбувається під капотом:

1. Викликається `__enter__()` → ресурс відкривається
2. Виконується блок `with`
3. Викликається `__exit__()` → ресурс **гарантовано звільняється**

Навіть якщо всередині блоку виникає виняток.

---

### 🧠 Архітектурна ідея

`with` вводить важливий принцип:

> **Життєвий цикл ресурсу прив’язаний до області виконання (scope), а не до логіки коду.**

* Вхід у блок → `acquire`
* Вихід з блоку → `release`
* Неважливо, як саме відбувся вихід (нормально чи через виняток)

---

### 🔧 Інсайт

`with` — це не просто синтаксичний цукор для `try...finally`.

Це:

* **контракт життєвого циклу**
* **інваріант системи**: ресурс не може “забутись”
* **зменшення когнітивного навантаження** (менше ручного контролю)

---

- > `with` створює чітку **просторову межу** (блок коду).
- > Вихід з межі = гарантоване очищення. Завжди.
---

In [ ]:
# Демонстрація: чому ручне close() ненадійне

# --- Небезпечний код ---
f = open('demo.txt', 'w')
try:
    f.write('рядок 1\n')
    # Уявімо: тут стався виняток
    # Без finally — f.close() ніколи не викличеться!
finally:
    f.close()
    print('Файл закрито вручну (try/finally)')

print('---')

# --- Безпечний код через with ---
with open('demo.txt', 'w') as f:
    f.write('рядок 1\n')
    f.write('рядок 2\n')
# Тут Python автоматично викликав f.__exit__() → f.close()

print(f'Файл закрито? {f.closed}')  # True — гарантовано

## Частина 2: Протокол `__enter__` та `__exit__` — Механіка «під капотом»

---

### Пояснення: Два dunder-методи, що утворюють протокол

Щоб об’єкт міг працювати з `with`, йому **не потрібно успадковувати жоден базовий клас**.
Python використовує *duck typing*: достатньо реалізувати два спеціальні (dunder) методи.

| Метод                                       | Коли викликається             | Що робить                                    |
| ------------------------------------------- | ----------------------------- | -------------------------------------------- |
| `__enter__(self)`                           | При **вході** в блок `with`   | Захоплює ресурс і повертає значення для `as` |
| `__exit__(self, exc_type, exc_val, exc_tb)` | При **виході** з блоку `with` | Звільняє ресурс і може обробити виняток      |

---

### Покрокова анатомія виконання `with expression as var:`

1. **Оцінка виразу**
   Python обчислює `expression`, отримуючи об’єкт контекстного менеджера.

2. **Вхід у контекст**
   Викликається `__enter__()`.

3. **Прив’язка змінної**
   Значення, яке повернув `__enter__()`, присвоюється `var`.

   > 🔍 **Нюанс:** `__enter__()` не зобов’язаний повертати `self`.
   > Він може повернути будь-який об’єкт (наприклад, обгортку, ресурс або адаптер).

4. **Виконання тіла блоку**
   Виконується код усередині `with`.

5. **Вихід з контексту (гарантований)**
   Викликається `__exit__()` **у будь-якому випадку**:

   * нормальне завершення
   * `return`, `break`, `continue`
   * виняток

6. **Обробка винятку**
   Якщо всередині блоку виник виняток:

 * `__exit__(exc_type, exc_val, exc_tb)` отримує інформацію про виняток
 * якщо `__exit__` повертає `True` → виняток **пригнічується**
 * якщо `False` або `None` → виняток **передається далі** (propagates)
---

### Візуальна модель виконання

```
with cm_object as resource:
    ┌────────────────────────────────────────┐
    │  cm_object.__enter__() викликано       │  ← setup
    │  resource = результат __enter__        │
    │  ... ваш код ...                       │
    │  cm_object.__exit__(exc_type, ...)     │  ← teardown (ЗАВЖДИ)
    └────────────────────────────────────────┘
```
`cm` — це скорочення від **context manager** (контекстний менеджер).

---


### 🧠 Важливий нюанс (часто пропускають)

Якщо **виняток виник у `__enter__()`**, тоді:

* `__exit__()` **не буде викликаний**
* тому вся логіка очищення має бути гарантована лише після успішного `__enter__`

---

### 🔧 Інсайт

Контекстний менеджер — це фактично:

* **двофазний протокол**:

  * `setup` → `__enter__`
  * `teardown` → `__exit__`

* з гарантією:

  > teardown виконається завжди, якщо setup завершився успішно

---

In [ ]:
# Мінімальний клас-менеджер контексту — «оголена механіка»

class SimpleContext:
    """Демонструє послідовність викликів __enter__ та __exit__."""

    def __enter__(self):
        # Фаза SETUP: тут захоплюємо ресурс
        print('__enter__: входимо в контекст')
        return self  # Повертаємо self → присвоюється змінній після 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Фаза TEARDOWN: тут звільняємо ресурс
        # exc_type, exc_val, exc_tb = None, None, None — якщо помилок не було
        print(f'__exit__: виходимо з контексту')
        print(f'  exc_type={exc_type}, exc_val={exc_val}')
        return False  # Не пригнічуємо виняток — нехай розповсюджується


print('=== Нормальне завершення ===')
with SimpleContext() as ctx:
    print(f'  всередині блоку, ctx={ctx}')

print()
print('=== Завершення з винятком ===')
try:
    with SimpleContext() as ctx:
        print('  робимо щось...')
        raise ValueError('щось пішло не так!')
except ValueError as e:
    print(f'  виняток перехоплено зовні: {e}')

## Частина 3: Паттерн Транзакції — `ListTransaction`

---

### Пояснення: «Все або нічого» — Unit of Work

Уявіть: ви модифікуєте список або базу даних.
Якщо під час змін виникає помилка, ви можете отримати **частково оновлені та неконсистентні дані**.

Це одна з найнебезпечніших категорій помилок —
дані виглядають «майже правильними», але вже порушують інваріанти системи.

---

### Рішення: транзакційний підхід

Патерн **Транзакція (Transaction / Unit of Work)** вирішує це через механізм:

> **«або всі зміни застосовані, або жодна» (all-or-nothing)**

---

### Як це працює

* Усі зміни виконуються над **робочою копією (working copy)**, а не над оригіналом
* Оригінальні дані залишаються **незмінними** до моменту підтвердження

| Етап                     | Метод       | Дія                                              |
| ------------------------ | ----------- | ------------------------------------------------ |
| Початок                  | `__enter__` | Створює робочу копію та повертає її              |
| Завершення (без помилки) | `__exit__`  | **Commit** → переносить зміни в оригінал         |
| Завершення (з помилкою)  | `__exit__`  | **Rollback** → відкидає зміни (копія знищується) |

---

### Логіка всередині `__exit__`

```python
if exc_type is None:
    # commit
else:
    # rollback
```

---

### 🧠 Архітектурний сенс

Контекстний менеджер тут виступає як:

* **межа транзакції**
* **гарант консистентності**
* **контролер життєвого циклу змін**

---

### 🔥 Важливий інсайт

Цей патерн гарантує:

* відсутність **часткових змін**
* збереження **інваріантів системи**
* передбачувану поведінку навіть при помилках

---

### 📌 Реальні приклади

Це не теорія — це стандарт у production:

* ORM: `sqlalchemy.orm.Session`
* стандартна бібліотека: `sqlite3` (`commit` / `rollback`)
* файлові системи (тимчасові файли → rename)

---


In [ ]:
class ListTransaction:
    """Контекстний менеджер, що реалізує паттерн Транзакції для списку."""

    def __init__(self, thelist):
        self.thelist = thelist  # Оригінальний список — захищений

    def __enter__(self):
        # SETUP: створюємо ізольовану робочу копію
        # Клієнт отримує КОПІЮ, а не сам оригінал
        self.workingcopy = list(self.thelist)
        return self.workingcopy

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            # Успіх → COMMIT: замінюємо вміст оригіналу вмістом копії
            # list[:] = ... модифікує оригінальний об'єкт на місці (in-place)
            self.thelist[:] = self.workingcopy
            print('COMMIT: зміни зафіксовано')
        else:
            # Помилка → ROLLBACK: просто не робимо нічого
            # Робоча копія просто знищиться збирачем сміття
            print(f'ROLLBACK: помилка {exc_type.__name__} — оригінал збережено')

        # Повертаємо False → не пригнічуємо виняток
        return False

In [ ]:
# === Сценарій 1: Успішна транзакція (Commit) ===
items = [1, 2, 3]
print(f'До: {items}')

with ListTransaction(items) as working:
    working.append(4)
    working.append(5)
    # Помилок не було → __exit__ отримає exc_type=None → Commit

print(f'Після: {items}')  # [1, 2, 3, 4, 5]

print()

# === Сценарій 2: Транзакція з помилкою (Rollback) ===
items = [1, 2, 3]
print(f'До: {items}')

try:
    with ListTransaction(items) as working:
        working.append(4)
        working.append(5)
        raise RuntimeError('Раптова помилка мережі!')  # Імітуємо збій
except RuntimeError:
    pass

# Зміни відкотились — список залишився незмінним
print(f'Після: {items}')  # [1, 2, 3] — захищено!

## Частина 4: Паттерн Тимчасового Стану — `RedirectStdout`

---

### Пояснення: Захист глобального стану

Іноді системі потрібно **тимчасово змінити глобальний або процесний стан** —
рівень логування, точність обчислень, стандартний потік виводу — виконати операцію,
а потім **гарантовано повернути все до попереднього стану**.

Без контекстного менеджера це небезпечно: якщо код між «змінити» і «відновити» завершиться помилкою —
система може залишитися в **некоректному або неконсистентному стані**.

---

### Схема патерну (Save → Modify → Restore)

* `__enter__` → **зберігає попередній стан** і встановлює новий
* `__exit__` → **гарантовано відновлює попередній стан** (навіть при виникненні винятку)

---

### 🧠 Архітектурний сенс

Цей патерн забезпечує:

* **ізоляцію змін** (зміни діють лише в межах `with`)
* **відсутність “витоків стану”** (state leakage)
* **детермінованість поведінки системи**

---

### 🔥 Інсайт

Контекстний менеджер тут працює як:

> **"знімок стану → тимчасова мутація → гарантоване відновлення"**

Це аналог:

* push/pop у стеку станів
* savepoint у транзакціях

---

### 📌 Реальні приклади зі стандартної бібліотеки

* `contextlib.redirect_stdout` — тимчасово перенаправляє `sys.stdout`
* `decimal.localcontext()` — змінює точність обчислень лише в межах блоку
* `unittest.mock.patch` — тимчасово підміняє об’єкти (dependency override)

---


In [ ]:
import sys
import io


class RedirectStdout:
    """Тимчасово перенаправляє sys.stdout на інший потік."""

    def __init__(self, new_target):
        self.new_target = new_target  # Новий потік виводу

    def __enter__(self):
        # SETUP: зберігаємо поточний stdout і замінюємо його
        self.original_stdout = sys.stdout  # Зберігаємо старий стан
        sys.stdout = self.new_target       # Встановлюємо новий стан
        return self.new_target

    def __exit__(self, exc_type, exc_val, exc_tb):
        # TEARDOWN: гарантовано відновлюємо оригінальний stdout
        # Навіть якщо всередині блоку стався виняток!
        sys.stdout = self.original_stdout
        return False

In [ ]:
# Перехоплюємо вивід у рядок замість терміналу
buffer = io.StringIO()

with RedirectStdout(buffer):
    print('Це піде в буфер, а не в термінал')
    print('І це теж')

# Після виходу з блоку — stdout відновлено
print('А це вже знову в термінал!')

# Отримуємо перехоплений текст
captured = buffer.getvalue()
print(f'Вміст буфера: {repr(captured)}')

# Python сам має contextlib.redirect_stdout — покажемо для порівняння
from contextlib import redirect_stdout
buf2 = io.StringIO()
with redirect_stdout(buf2):
    print('Стандартна бібліотека робить те саме!')
print(f'buf2: {repr(buf2.getvalue())}')

## Частина 5: `@contextmanager` — Елегантність на основі генераторів

---

### Пояснення: Як `yield` «розділяє» функцію в контекстному менеджері

Створення окремого класу з `__enter__` та `__exit__` для кожної невеликої задачі —
часто надмірне і порушує принцип лаконічності.

Модуль `contextlib` і декоратор `@contextmanager` дозволяють описати
контекстний менеджер як **функцію-генератор**.

---

### Як це працює

Ключове слово `yield` **логічно розділяє функцію на дві частини**:

```python id="ctx_gen_split"
@contextmanager
def managed_resource():
    ┌─────────────────────────────────────┐
    │  код до yield      → __enter__      │  ← захоплення ресурсу
    │  yield value       → as value       │  ← передача в блок with
    │  код після yield   → __exit__       │  ← звільнення ресурсу
    └─────────────────────────────────────┘
```

---

### Що відбувається під капотом

1. Викликається функція → повертається генератор
2. Виконується код **до `yield`** → це аналог `__enter__`
3. Значення з `yield` передається в `as`
4. Після завершення блоку `with`:

   * генератор **відновлюється**
   * виконується код **після `yield`** → це аналог `__exit__`

---

### ⚠️ Архітектурне правило: обробка винятків

Якщо всередині блоку `with` виникає виняток:

> Python **повертає його назад у генератор — у точку `yield`**

Тобто цей код:

```python id="ctx_error_flow"
yield resource
```

фактично стає місцем, куди “повертається” виняток.

---

### ❗ Критично важливо

Якщо не обгорнути `yield` у `try/finally`:

* код після `yield` може **не виконатися коректно**
* ви втратите гарантію **teardown (cleanup)**

---

### ✔️ Правильний шаблон

```python id="ctx_best_practice"
from contextlib import contextmanager

@contextmanager
def managed_resource():
    resource = acquire()
    try:
        yield resource
    finally:
        release(resource)
```

---

### 🧠 Інсайт

`@contextmanager` — це:

* **синтаксичний адаптер** генератора → у протокол `__enter__/__exit__`
* але відповідальність за:

  * cleanup
  * обробку винятків
    залишається **на розробнику**

Тому `try/finally` навколо `yield` є **архітектурним обов'язком**.
---


In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def timethis(label):
    """Вимірює час виконання блоку коду."""
    # === Аналог __enter__ ===
    start = time.time()  # Фіксуємо час початку
    try:
        yield  # Передаємо управління тілу блоку with (значення None)
               # Функція «заморожується» тут, поки блок виконується
    finally:
        # === Аналог __exit__ ===
        # finally гарантує виконання навіть при помилці в блоці
        end = time.time()
        print(f'{label}: {end - start:.4f} сек')


# Використання:
with timethis('Зворотній відлік'):
    n = 5_000_000
    while n > 0:
        n -= 1

In [ ]:
from contextlib import contextmanager


@contextmanager
def managed_file(name):
    """Безпечно відкриває файл і гарантує його закриття."""
    # === __enter__: відкриваємо ресурс ===
    f = open(name, 'w', encoding='utf-8')
    try:
        yield f  # Об'єкт файлу передається в змінну 'as'
    finally:
        # === __exit__: ЗАВЖДИ закриваємо файл ===
        f.close()
        print(f'Файл "{name}" закрито гарантовано')


with managed_file('output.txt') as f:
    f.write('привіт, світ!\n')
    f.write('а тепер, бувай!\n')

In [ ]:
# === Що відбувається, коли НЕМАЄ try/finally? ===
# Демонстрація архітектурної помилки — небезпечний варіант

from contextlib import contextmanager


@contextmanager
def dangerous_managed_file(name):
    """НЕБЕЗПЕЧНО: немає try/finally — teardown не гарантований!"""
    f = open(name, 'w', encoding='utf-8')
    yield f
    # Якщо тут виникне помилка — f.close() НЕ ВИКЛИЧЕТЬСЯ!
    f.close()
    print('Цей рядок не виконається при помилці!')


try:
    with dangerous_managed_file('danger.txt') as f:
        f.write('записуємо...\n')
        raise IOError('Симуляція збою диска!')
except IOError as e:
    print(f'Помилка: {e}')
    print(f'Файл закрито? {f.closed}')  # False — витік ресурсу!

## Частина 6: Lock Pattern — Уникнення Взаємного Блокування

---

## 🔴 Deadlock — зупинка через взаємне блокування

У багатопотоковому середовищі керування м’ютексами (`lock`) є критично чутливим.
Неправильна робота з блокуваннями може призвести до **deadlock (взаємного блокування)** — стану, коли потоки **назавжди очікують один одного**, і система фактично перестає виконуватись.

---

## ✔️ Базове правило: безпечне звільнення блокування

Використання:

```python
with lock:
    # критична секція
```

гарантує, що:

* блокування буде **звільнене завжди**
* навіть якщо всередині виникне виняток

Еквівалент низького рівня:

```python
lock.acquire()
try:
    ...
finally:
    lock.release()
```

---

## ⚠️ Важливо

`with lock`:

* ✅ вирішує проблему **забутого `release()`**
* ❌ **не вирішує проблему deadlock між кількома lock**

---

## 🔁 Класичний deadlock (circular wait)

Deadlock виникає при **циклічному очікуванні ресурсів**:

* Потік A:

  * захоплює `lock_1`
  * чекає `lock_2`

* Потік B:

  * захоплює `lock_2`
  * чекає `lock_1`

→ виникає **кругове очікування (circular wait)**
→ жоден потік не може продовжити виконання

---

## 🧠 Архітектурне рішення: глобальний порядок блокувань

> **Усі блокування мають захоплюватися в однаковому порядку**

```python
locks = sorted([lock1, lock2], key=id)

for lock in locks:
    lock.acquire()

try:
    ...
finally:
    for lock in reversed(locks):
        lock.release()
```

---

## 🔬 Чому це працює

```log
1. Усі потоки дотримуються одного порядку
2. Неможливо сформувати цикл очікування
3. → усувається circular wait
4. → deadlock стає неможливим
```

---

## 🧩 Що таке м’ютекс (mutex)?

**М’ютекс (mutex, mutual exclusion)** — це примітив синхронізації, який гарантує:

> **лише один потік може виконувати критичну секцію в один момент часу**

---

## 🧠 Інтуїція

М’ютекс = **замок на ресурсі**

* `acquire()` → «зачинити двері»
* `release()` → «відчинити двері»
* інші потоки → чекають

---

## 📌 У Python

```python
import threading

lock = threading.Lock()

with lock:
    print("Тільки один потік тут")
```

---

## ⚠️ Проблема без м’ютекса (race condition)

```python
balance = 100

def withdraw():
    global balance
    temp = balance
    temp -= 10
    balance = temp
```

При паралельному виконанні:
→ значення може стати некоректним

---

## ✔️ Рішення

```python
lock = threading.Lock()

def withdraw():
    global balance
    with lock:
        balance -= 10
```

→ операція стає **захищеною (thread-safe)**

---

## 🔥 Ключові терміни

| Термін           | Значення                                 |
| ---------------- | ---------------------------------------- |
| Critical section | код, який не можна виконувати паралельно |
| Acquire          | захопити блокування                      |
| Release          | звільнити блокування                     |
| Race condition   | помилка через одночасний доступ          |
| Deadlock         | взаємне блокування потоків               |

---

## 🧠 Інсайт

М’ютекс — це:

* контроль доступу до **спільного стану**
* гарантія **консистентності**
* фундамент для:

  * транзакцій
  * конкурентних систем
  * черг та scheduler-ів

---

## ⚖️ Trade-off

```log
[FACT] Mutex ≠ продуктивність
[FACT] Mutex = безпека
[TRADEOFF] більше блокувань → менше паралелізму
```

---

## 🔗 Зв’язок з контекстними менеджерами

```python
with lock:
```

→ це і є **ідеальний інтерфейс для mutex у Python**

Інтерпретатор гарантує:

* правильний життєвий цикл (`acquire → release`)
* відсутність людського фактору

---

In [ ]:
import threading


class SharedCounter:
    """Потокобезпечний лічильник через контекстний менеджер."""

    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()  # М'ютекс для захисту стану

    def increment(self):
        # 'with lock' — це паттерн «Вікно ексклюзивного доступу»
        # Блокування знімається ГАРАНТОВАНО при виході з блоку
        with self._lock:
            self._value += 1
        # Тут блокування вже знято — інші потоки можуть продовжити

    @property
    def value(self):
        with self._lock:  # Навіть читання захищаємо для thread-safety
            return self._value


# Тест: 5 потоків по 1000 інкрементів = рівно 5000
counter = SharedCounter()
threads = [threading.Thread(target=lambda: [counter.increment() for _ in range(1000)])
           for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f'Результат: {counter.value}')  # Завжди 5000 — race condition виключено

In [ ]:
import threading
from contextlib import contextmanager
import time


@contextmanager
def acquire(*locks):
    """
    Захоплює кілька блокувань у строго фіксованому порядку.

    Математичне доведення відсутності deadlock:
    Якщо всі потоки захоплюють блокування в одному й тому ж порядку
    (за id об'єкта), кругове очікування стає неможливим.
    """
    # Сортуємо блокування за їх ідентифікатором в пам'яті
    # Це гарантує однаковий порядок незалежно від аргументів
    sorted_locks = sorted(locks, key=lambda x: id(x))

    acquired = []
    try:
        for lock in sorted_locks:
            lock.acquire()       # Захоплюємо по одному у суворому порядку
            acquired.append(lock)
        yield  # Передаємо управління критичній секції
    finally:
        # Звільняємо у зворотному порядку (LIFO)
        for lock in reversed(acquired):
            lock.release()


# Тест: навіть якщо потоки просять блокування в різному порядку,
# acquire() примусово впорядковує їх → deadlock неможливий

x = threading.Lock()
y = threading.Lock()

def worker1():
    with acquire(x, y):
        print("Worker 1 acquired x, y")
        time.sleep(1)

def worker2():
    with acquire(y, x):
        print("Worker 2 acquired y, x")
        time.sleep(1)

t1 = threading.Thread(target=worker1)
t2 = threading.Thread(target=worker2)

t1.start()
t2.start()

t1.join()
t2.join()

## Частина 7: Transaction Pattern — E-Commerce Резервування

---

### Пояснення: Патерн Cleanup — запобігання помилкам через пропущене очищення

Уявіть систему, де замовлення **резервує товари на складі**.
Якщо оплата картою не пройде — ми **зобов’язані скасувати резервування**, інакше система залишиться в неконсистентному стані.

---

### ❗ Проблема без контекстного менеджера

Якщо код виглядає так:

```python
reserve_items(order, inventory)
charge_credit_card(order)
unreserve_items(order, inventory)
```

і `charge_credit_card()` кидає виняток:

→ `unreserve_items()` **не викличеться**
→ товари залишаться «заблокованими»
→ виникає **логічна помилка в даних**

---

### ✔️ Рішення: контекстний менеджер

Контекстний менеджер інкапсулює cleanup-логіку в самому об’єкті:

```python
with create_grocery_list(order, inventory) as lst:
    charge_credit_card(order)
```

---

### 🔁 Що відбувається

* `__enter__` → резервує товари
* тіло `with` → виконує бізнес-логіку (оплата)
* `__exit__` →

  * якщо помилки **не було** → commit (залишає резерв)
  * якщо помилка **була** → викликає `unreserve_items()`

👉 cleanup виконується **гарантовано**

---

### 🧠 Архітектурний сенс

Це приклад патерну:

> **Cleanup / Resource Safety / RAII (в Python-стилі)**

Контекстний менеджер:

* усуває **людський фактор**
* гарантує виконання критичних дій
* переносить відповідальність з розробника → у систему

---

### 🔥 Інсайт

> **Правильний дизайн робить помилки неможливими, а не просто малоймовірними**

---

### 📌 Важливе уточнення

Цей патерн:

* не лише про файли чи lock-и
* а про **будь-яку операцію з побічними ефектами**:

  * резервування
  * відкриття з’єднання
  * зміну стану системи

---

In [ ]:
from contextlib import contextmanager


class GroceryList:
    """Список зарезервованих товарів для замовлення."""

    def __init__(self, order_id, items):
        self.order_id = order_id
        self._reserved = []  # Зарезервовані позиції

        # Резервуємо товари при створенні
        for item in items:
            self._reserved.append(item)
            print(f'  [WAREHOUSE] Зарезервовано: {item}')

    def has_reserved_items(self):
        return bool(self._reserved)

    def unreserve_items(self):
        print(f'  [WAREHOUSE] Знімаємо резерв для замовлення {self.order_id}')
        self._reserved.clear()


@contextmanager
def create_grocery_list(order_id, items):
    """Резервує товари і гарантує скасування при помилці."""
    grocery_list = GroceryList(order_id, items)
    try:
        yield grocery_list  # Передаємо список клієнтському коду
    finally:
        # Очищення: якщо товари досі зарезервовані після виходу — скасовуємо
        if grocery_list.has_reserved_items():
            grocery_list.unreserve_items()


# === Сценарій: Оплата не пройшла ===
print('Обробка замовлення #42...')
try:
    with create_grocery_list('#42', ['яблука', 'молоко', 'хліб']) as order:
        print('  [PAYMENT] Спроба оплати...')
        raise ConnectionError('Платіжний шлюз недоступний!')  # Імітуємо збій
except ConnectionError as e:
    print(f'  [ERROR] {e}')

print('Резервування скасовано автоматично — склад вільний!')

## Частина 8: `__exit__` як «контролер» — управління винятками
---

### Пояснення: Булевий перемикач — «пропустити» чи «придушити»

Коли всередині блоку `with` виникає виняток, Python **не піднімає його одразу**.
Спочатку він викликає `__exit__`, передаючи інформацію про помилку:

> *«Виняток стався. Чи вважаєш ти його обробленим?»*

---

### 📌 Аргументи `__exit__`

| Аргумент   | Значення при винятку                | Значення без винятку |
| ---------- | ----------------------------------- | -------------------- |
| `exc_type` | Клас винятку (`ZeroDivisionError`)  | `None`               |
| `exc_val`  | Екземпляр винятку (з повідомленням) | `None`               |
| `exc_tb`   | Об’єкт трасування (traceback)       | `None`               |

---

### ✔️ Що повертає `__exit__`

* **`False` або `None`** → виняток **передається далі по стеку викликів** (нормальна поведінка)
* **`True`** → виняток вважається **обробленим і пригнічується**
  → виконання продовжується з першого рядка **після `with`**

---

### 🧠 Важливий нюанс

Якщо **винятку не було**:

```python id="no_exc"
__exit__(None, None, None)
```

→ значення, яке повертає `__exit__`, **ігнорується**

---

### ⚠️ Антипатерн: «Чорна діра»

Ніколи не пишіть:

```python id="black_hole"
def __exit__(...):
    return True
```

без перевірки `exc_type`.

👉 Інакше ви:

* пригнічуєте **всі винятки**
* включно з:

  * `KeyboardInterrupt`
  * `SystemExit`
  * критичними runtime-помилками

→ система стає **непрозорою і небезпечною для дебагу**

---

### ✔️ Правильний підхід

```python id="safe_exit"
def __exit__(self, exc_type, exc_val, exc_tb):
    if exc_type is SomeExpectedError:
        handle()
        return True  # пригнічуємо тільки очікувану помилку
    return False
```

---

### 🔥 Інсайт

`__exit__` — це не просто cleanup.

Це:

* **фільтр винятків**
* **контроль точки завершення**
* **механізм політики обробки помилок**

---

In [ ]:
class ErrorSuppressor:
    """Вибірково пригнічує конкретний тип винятку."""

    def __init__(self, target_exception):
        self.target = target_exception  # Який тип помилки нас цікавить

    def __enter__(self):
        print('Входимо в контекст...')
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'Виходимо з контексту (exc_type={exc_type})')

        # Якщо помилок не було — нічого не робимо
        if exc_type is None:
            return False

        # Якщо помилка збігається з нашим цільовим типом — пригнічуємо
        if issubclass(exc_type, self.target):
            print(f'  [SUPPRESSED] {exc_type.__name__}: {exc_val}')
            return True  # ← Виняток знищено тут!

        # Усі інші помилки — пропускаємо далі
        return False


# === Тест 1: Придушення ZeroDivisionError ===
print('=== Тест 1: Ділення на нуль ===')
with ErrorSuppressor(ZeroDivisionError):
    print('Обчислення...')
    result = 10 / 0          # Виняток!
    print('Цей рядок ніколи не виконається')  # Пропущено
print('Програма продовжується після блоку — виняток знищено!\n')

# === Тест 2: Незнайомий тип — пропускаємо далі ===
print('=== Тест 2: ValueError — не придушуємо ===')
try:
    with ErrorSuppressor(ZeroDivisionError):
        raise ValueError('Невідома помилка!')
except ValueError as e:
    print(f'  ValueError впав назовні: {e}')

## Частина 9: `contextlib.suppress` — Стандартна Бібліотека в Дії

---

### Пояснення: `contextlib.suppress` — готовий «пригнічувач винятків»

Стандартна бібліотека Python реалізує той самий механізм через `contextlib.suppress`.
Це елегантна заміна громіздким конструкціям `try: ... except: pass`.

---

### ✔️ Порівняння підходів

```python id="old_style"
# Старий стиль
try:
    os.remove('tempfile.txt')
except FileNotFoundError:
    pass
```

```python id="new_style"
# Новий стиль — чистіше і читабельніше
with contextlib.suppress(FileNotFoundError):
    os.remove('tempfile.txt')
```

---

### 🧠 Що відбувається під капотом

`contextlib.suppress(FileNotFoundError)`:

* створює контекстний менеджер
* всередині `__exit__`:

  * перевіряє тип винятку
  * якщо це `FileNotFoundError` → **повертає `True` (пригнічує)**
  * інакше → **передає виняток далі**

---

### 🔥 Інсайт

> `suppress` — це **безпечна версія `except: pass`**,
> яка пригнічує **лише очікувані винятки**

---

### ⚠️ Важливе обмеження

```python id="danger"
with contextlib.suppress(Exception):
    ...
```

👉 технічно працює, але це **антипатерн**

* пригнічує **всі помилки**
* ускладнює дебаг
* може приховати критичні збої

---

### ✔️ Правильний підхід

```python id="best"
with contextlib.suppress(FileNotFoundError, PermissionError):
    ...
```

👉 явно вказуєш, **які саме помилки очікувані**

---

In [ ]:
import contextlib
import os

# Видаляємо файл, якщо він є — ігноруємо якщо його немає
with contextlib.suppress(FileNotFoundError):
    os.remove('nonexistent_file.txt')  # Не впаде — помилка придушена
print('Файл видалено (або його й не було — нам байдуже)')

print()

# Можна придушувати кілька типів одразу
with contextlib.suppress(FileNotFoundError, PermissionError):
    os.remove('/root/protected_file.txt')
print('Ніяких падінь, навіть при відмові доступу')

print()

# decimal.localcontext() — тимчасова точність без зміни глобальних налаштувань
import decimal

with decimal.localcontext() as ctx:
    ctx.prec = 50  # Встановлюємо локальну точність на 50 знаків
    result = decimal.Decimal(1) / decimal.Decimal(3)
    print(f'Локальна точність (50 знаків): {result}')

# Після виходу — глобальна точність відновлена
print(f'Глобальна точність: {decimal.Decimal(1) / decimal.Decimal(3)}')

## Підсумок: Таблиця Архітектурних Паттернів

---

| Паттерн | `__enter__` | `__exit__` | Приклад |
|---------|-------------|------------|---------|
| **Resource Cleanup** | Відкрити файл / сокет | `close()` | `open()`, сокети |
| **Lock / Sync** | `lock.acquire()` | `lock.release()` | `threading.Lock` |
| **Transaction** | Створити робочу копію | Commit або Rollback | SQLAlchemy, `sqlite3` |
| **Temporary State** | Зберегти і замінити стан | Відновити старий стан | `redirect_stdout`, `decimal.localcontext` |
| **Timer / Audit** | Зафіксувати початок | Зафіксувати кінець / лог | Профілювання, аудит |
| **E-commerce Reservation** | Зарезервувати ресурс | Скасувати при збої | Склад, квитки |

---

> **Ключова думка:**
> Контекстні менеджери — це не про файли.  
> Це фундамент для створення **надійних (robust) API**, де правильна поведінка системи —  
> очищення, відкат, звільнення ресурсів — **вбудована в саму структуру коду**  
> і не потребує ручного контролю.  
>  
> Ви створюєте інтерфейси, якими **дуже легко користуватись правильно**  
> і **майже неможливо скористатись неправильно**.